# Lab — Pixel-Level Industrial Inspection and Interactive Mask Refinement

This credential-free CPU lab connects Course 05 boxes to exact pixel masks. It trains a small educational U-Net, evaluates semantic and boundary evidence, and measures the sensitivity of a transparent promptable baseline.

The local prompt proxy is **not a foundation model**. It makes point/box mechanics executable without downloads. A separately governed SAM 3.1 adapter is opt-in and excluded from local claims.


## 0. Experiment contract

**Question:** what new evidence is required when an inspection system predicts exact pixels rather than boxes?

**Validated path:** deterministic synthetic A/B/C factories, NumPy/Pillow/SciPy mask operations, a PyTorch U-Net, local prompt proxy, and JSON evidence. No network, credentials, or pretrained weights.

**Success evidence:** mask IDs survive preprocessing; scratch metrics pass known cases; the U-Net returns full-resolution logits; CE/Dice/combined losses are compared; tiny/thin/low-contrast/occluded/source slices are reported; prompt and detector-box perturbations are measured.

**Boundary:** procedural evidence validates the learning workflow, not a production inspection claim. No cell authorizes physical action.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageDraw
import scipy
from scipy import ndimage, stats
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision

SEED = 23
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.use_deterministic_algorithms(True)
torch.set_num_threads(min(2, os.cpu_count() or 1))
DEVICE = torch.device("cpu")

VERSIONS = {
    "python": platform.python_version(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": PIL.__version__,
    "scipy": scipy.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
}
print(pd.Series(VERSIONS, name="version").to_string())
print(f"device={DEVICE}; deterministic=True; seed={SEED}")


## 1. Generate a source-aware pixel corpus

The corpus contains irregular component bodies, separate object instances, tiny contamination, thin damage, sealant-like boundaries, holes, partial occlusion, and ambiguous edges. Factory C changes colors, texture, and contrast and is held out from training.

The labels are five integer IDs: `0 background`, `1 component`, `2 contamination`, `3 surface_damage`, and `4 sealant`. Instance IDs are stored separately from semantic IDs.


In [ ]:
CLASS_NAMES = {0: "background", 1: "component", 2: "contamination", 3: "surface_damage", 4: "sealant"}
CLASS_COLORS = np.array([
    [20, 28, 40], [76, 175, 230], [244, 162, 97], [230, 70, 93], [112, 210, 160]
], dtype=np.uint8)
IMAGE_SIZE = 64


def _irregular_polygon(rng, cx, cy, radius, vertices=9):
    angles = np.linspace(0, 2 * np.pi, vertices, endpoint=False)
    angles += rng.uniform(-0.12, 0.12, size=vertices)
    radii = radius * rng.uniform(0.72, 1.16, size=vertices)
    return [(int(cx + r * np.cos(a)), int(cy + r * np.sin(a))) for a, r in zip(angles, radii)]


def make_sample(index: int, source: str, size: int = IMAGE_SIZE) -> dict:
    rng = np.random.default_rng(SEED * 1000 + index + {"A": 0, "B": 10000, "C": 20000}[source])
    low_contrast = index % 4 == 0
    occluded = index % 5 == 0
    boundary_heavy = index % 3 == 0
    yy, xx = np.mgrid[:size, :size]

    backgrounds = {"A": np.array([39, 51, 63]), "B": np.array([61, 45, 42]), "C": np.array([84, 82, 68])}
    base = backgrounds[source].astype(float)
    image = np.broadcast_to(base, (size, size, 3)).copy()
    image += (xx[..., None] / size - 0.5) * ({"A": 12, "B": -10, "C": 18}[source])
    if source == "C":
        image += 8 * np.sin((xx + yy) / 3.8)[..., None]
    image += rng.normal(0, 3.5 if source != "C" else 6.0, image.shape)

    semantic_pil = Image.new("L", (size, size), 0)
    instance_pil = Image.new("I", (size, size), 0)
    semantic_draw = ImageDraw.Draw(semantic_pil)
    instance_draw = ImageDraw.Draw(instance_pil)

    centers = [(31 + int(rng.integers(-4, 5)), 32 + int(rng.integers(-4, 5)), 20)]
    if index % 3 == 1:
        centers = [(22, 31, 13), (43, 33, 12)]
    polygons = []
    for instance_id, (cx, cy, radius) in enumerate(centers, start=1):
        polygon = _irregular_polygon(rng, cx, cy, radius, 11 if boundary_heavy else 8)
        polygons.append(polygon)
        semantic_draw.polygon(polygon, fill=1)
        instance_draw.polygon(polygon, fill=instance_id)

    semantic = np.asarray(semantic_pil, dtype=np.uint8).copy()
    instances = np.asarray(instance_pil, dtype=np.int32).copy()
    component = semantic == 1

    ring_pil = Image.new("L", (size, size), 0)
    ring_draw = ImageDraw.Draw(ring_pil)
    for polygon in polygons:
        ring_draw.line(polygon + [polygon[0]], fill=1, width=2, joint="curve")
    ring = (np.asarray(ring_pil) > 0) & component
    semantic[ring] = 4

    contamination_masks = []
    for blob_index in range(1 + index % 2):
        coords = np.argwhere(component)
        cy, cx = coords[int(rng.integers(0, len(coords)))]
        radius = 2 if (index + blob_index) % 3 else 4
        blob = (xx - cx) ** 2 + (yy - cy) ** 2 <= radius**2
        blob &= component
        contamination_masks.append(blob)
        semantic[blob] = 2
        instances[blob] = 10 + blob_index

    damage_pil = Image.new("L", (size, size), 0)
    damage_draw = ImageDraw.Draw(damage_pil)
    start = (int(rng.integers(15, 28)), int(rng.integers(18, 28)))
    end = (int(rng.integers(38, 53)), int(rng.integers(35, 49)))
    damage_draw.line([start, end], fill=1, width=1 if index % 2 else 2)
    damage = (np.asarray(damage_pil) > 0) & component
    semantic[damage] = 3

    if index % 6 == 0:
        hy, hx = np.argwhere(component)[len(np.argwhere(component)) // 3]
        hole = (xx - hx) ** 2 + (yy - hy) ** 2 <= 2**2
        semantic[hole] = 0
        instances[hole] = 0

    source_palette = {
        "A": np.array([[39, 51, 63], [160, 174, 186], [183, 117, 72], [205, 76, 84], [100, 181, 145]]),
        "B": np.array([[61, 45, 42], [178, 160, 143], [195, 131, 77], [202, 69, 92], [96, 176, 150]]),
        "C": np.array([[84, 82, 68], [124, 122, 105], [139, 112, 78], [151, 91, 88], [99, 142, 118]]),
    }[source].astype(float)
    if low_contrast:
        source_palette[1:] = 0.58 * source_palette[1:] + 0.42 * source_palette[0]
    for class_id in range(1, len(CLASS_NAMES)):
        image[semantic == class_id] = source_palette[class_id] + rng.normal(0, 4, (int((semantic == class_id).sum()), 3))

    if occluded:
        x0 = 27 + int(rng.integers(-3, 4))
        image[:, x0 : x0 + 5] *= 0.45
    image = np.clip(image, 0, 255).astype(np.uint8)

    class_areas = {CLASS_NAMES[k]: int((semantic == k).sum()) for k in CLASS_NAMES}
    return {
        "id": f"{source}-{index:03d}", "source": source, "image": image,
        "mask": semantic, "instances": instances,
        "low_contrast": low_contrast, "occlusion": occluded,
        "thin": int((semantic == 3).sum()) > 0,
        "tiny": class_areas["contamination"] < 55,
        "boundary_heavy": boundary_heavy,
        "class_areas": class_areas,
    }


train_samples = [make_sample(i, source) for source in ("A", "B") for i in range(24)]
test_samples = [make_sample(100 + i, source) for source in ("A", "B") for i in range(6)]
test_samples += [make_sample(200 + i, "C") for i in range(12)]
print(f"train={len(train_samples)} ({sorted(set(s['source'] for s in train_samples))}); test={len(test_samples)}")
assert set(np.unique(np.concatenate([s["mask"].ravel() for s in train_samples]))) == set(CLASS_NAMES)
assert all(s["image"].shape[:2] == s["mask"].shape == s["instances"].shape for s in train_samples + test_samples)


### Mask-quality report and visual contract

Connected components expose fragmented or touching regions that class area alone hides. “Overlap errors” would be checked before collapsing per-instance binary masks into one integer instance map; this generated map stores one visible identity per pixel, so overlap is zero by construction.


In [ ]:
def mask_quality_report(samples):
    rows = []
    for sample in samples:
        component_counts = {}
        tiny_components = 0
        for class_id, name in CLASS_NAMES.items():
            if class_id == 0:
                continue
            labeled, count = ndimage.label(sample["mask"] == class_id)
            component_counts[name] = int(count)
            for region_id in range(1, count + 1):
                tiny_components += int((labeled == region_id).sum() < 20)
        rows.append({
            "id": sample["id"], "source": sample["source"],
            "shape": str(sample["mask"].shape), "dtype": str(sample["mask"].dtype),
            "unique_ids": tuple(np.unique(sample["mask"]).tolist()),
            "foreground_fraction": float((sample["mask"] > 0).mean()),
            "connected_components": sum(component_counts.values()),
            "tiny_components": tiny_components, "overlap_errors": 0,
        })
    return pd.DataFrame(rows)


quality_report = mask_quality_report(train_samples + test_samples)
display(quality_report.groupby("source")[["foreground_fraction", "connected_components", "tiny_components", "overlap_errors"]].agg(["mean", "max"]).round(3))


def colorize(mask):
    return CLASS_COLORS[np.asarray(mask, dtype=np.int64)]


fig, axes = plt.subplots(3, 3, figsize=(10, 9))
for row, sample in enumerate([train_samples[0], train_samples[25], test_samples[-1]]):
    overlay = (0.62 * sample["image"] + 0.38 * colorize(sample["mask"])).astype(np.uint8)
    for col, (title, image) in enumerate((("image", sample["image"]), ("mask", colorize(sample["mask"])), ("overlay", overlay))):
        axes[row, col].imshow(image); axes[row, col].set_title(f"{sample['id']} · {title}"); axes[row, col].axis("off")
plt.tight_layout()


## 2. Resize interpolation is part of the label contract

Images are continuous measurements; class IDs are categorical symbols. Bilinear interpolation creates mixtures between IDs. Nearest-neighbor selects an existing label.


In [ ]:
toy_mask = torch.zeros(1, 1, 8, 8, dtype=torch.float32)
toy_mask[:, :, 2:7, 3:6] = 4.0
nearest = F.interpolate(toy_mask, size=(13, 13), mode="nearest")
bilinear = F.interpolate(toy_mask, size=(13, 13), mode="bilinear", align_corners=False)

nearest_values = torch.unique(nearest).tolist()
bilinear_values = torch.unique(bilinear).tolist()
print("nearest unique IDs:", nearest_values)
print("bilinear first values:", [round(v, 3) for v in bilinear_values[:12]], f"({len(bilinear_values)} unique)")
assert set(nearest_values) <= {0.0, 4.0}
assert any(not float(v).is_integer() for v in bilinear_values)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, title, value in zip(axes, ["original IDs", "nearest: valid", "bilinear: corrupted"], [toy_mask[0, 0], nearest[0, 0], bilinear[0, 0]]):
    im = ax.imshow(value, vmin=0, vmax=4, cmap="viridis"); ax.set_title(title); ax.axis("off")
fig.colorbar(im, ax=axes, fraction=0.025); plt.show()


## 3. Implement segmentation metrics from scratch

All empty-set policies are explicit. For a class absent in both prediction and target, per-class IoU returns `NaN`; mIoU uses `nanmean` over present classes. Change that policy only deliberately.


In [ ]:
def binary_iou(pred, target):
    """Binary IoU: empty/empty=1; exactly one empty=0."""
    pred, target = np.asarray(pred, bool), np.asarray(target, bool)
    intersection = np.logical_and(pred, target).sum()
    union = np.logical_or(pred, target).sum()
    return 1.0 if union == 0 else float(intersection / union)


def dice_score(pred, target):
    """Binary Dice: empty/empty=1; exactly one empty=0."""
    pred, target = np.asarray(pred, bool), np.asarray(target, bool)
    intersection = np.logical_and(pred, target).sum()
    denominator = pred.sum() + target.sum()
    return 1.0 if denominator == 0 else float(2 * intersection / denominator)


def pixel_accuracy(pred, target):
    pred, target = np.asarray(pred), np.asarray(target)
    assert pred.shape == target.shape
    return float((pred == target).mean())


def per_class_iou(pred, target, class_ids):
    """Absent/absent classes are NaN so macro averaging can declare its policy."""
    result = {}
    for class_id in class_ids:
        p, t = np.asarray(pred) == class_id, np.asarray(target) == class_id
        union = np.logical_or(p, t).sum()
        result[int(class_id)] = float(np.logical_and(p, t).sum() / union) if union else np.nan
    return result


def mean_iou(pred, target, class_ids):
    return float(np.nanmean(list(per_class_iou(pred, target, class_ids).values())))


def foreground_recall(pred, target):
    p, t = np.asarray(pred) > 0, np.asarray(target) > 0
    return float(np.logical_and(p, t).sum() / max(t.sum(), 1))


perfect = np.array([[0, 1], [1, 0]])
disjoint = np.array([[1, 0], [0, 1]])
partial = np.array([[0, 1], [0, 0]])
assert binary_iou(perfect, perfect) == 1.0
assert dice_score(perfect, perfect) == 1.0
assert binary_iou(perfect, disjoint) == 0.0
partial_iou = binary_iou(partial, perfect)
assert np.isclose(dice_score(partial, perfect), 2 * partial_iou / (1 + partial_iou))

imbalance_target = np.zeros((10, 10), dtype=np.uint8); imbalance_target[0, 0] = 1
all_background = np.zeros_like(imbalance_target)
print({"pixel_accuracy": pixel_accuracy(all_background, imbalance_target), "defect_recall": foreground_recall(all_background, imbalance_target)})
assert pixel_accuracy(all_background, imbalance_target) == 0.99
assert foreground_recall(all_background, imbalance_target) == 0.0


## 4. Boundary F1 makes edge tolerance executable

We extract a one-pixel inner contour and allow matching within `tolerance` pixels by dilation. Boundary precision asks how much predicted contour lies near the target; boundary recall asks how much target contour lies near the prediction.


In [ ]:
def mask_boundary(mask):
    mask = np.asarray(mask, bool)
    if not mask.any():
        return np.zeros_like(mask)
    return np.logical_xor(mask, ndimage.binary_erosion(mask))


def boundary_f1(pred, target, tolerance=1, eps=1e-8):
    pred_b, target_b = mask_boundary(pred), mask_boundary(target)
    if not pred_b.any() and not target_b.any():
        return 1.0
    if not pred_b.any() or not target_b.any():
        return 0.0
    structure = ndimage.generate_binary_structure(2, 1)
    pred_near = ndimage.binary_dilation(pred_b, structure=structure, iterations=tolerance)
    target_near = ndimage.binary_dilation(target_b, structure=structure, iterations=tolerance)
    precision = np.logical_and(pred_b, target_near).sum() / max(pred_b.sum(), 1)
    recall = np.logical_and(target_b, pred_near).sum() / max(target_b.sum(), 1)
    return float((2 * precision * recall + eps) / (precision + recall + eps))


target = np.zeros((32, 32), bool); target[7:25, 7:25] = True
smooth_shift = np.zeros_like(target); smooth_shift[8:26, 7:25] = True
jagged = target.copy(); jagged[7:25:2, 23:27] = True; jagged[8:25:2, 21:25] = False
assert np.isclose(boundary_f1(target, target, tolerance=1), 1.0)
assert boundary_f1(np.zeros_like(target), target, tolerance=1) == 0.0

boundary_examples = pd.DataFrame([
    {"prediction": "smooth shift", "IoU": binary_iou(smooth_shift, target), "boundary_F1": boundary_f1(smooth_shift, target, 1)},
    {"prediction": "jagged edge", "IoU": binary_iou(jagged, target), "boundary_F1": boundary_f1(jagged, target, 1)},
])
display(boundary_examples.round(3))
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, name, value in zip(axes, ["target", "smooth shift", "jagged"], [target, smooth_shift, jagged]):
    ax.imshow(value, cmap="gray"); ax.set_title(name); ax.axis("off")
plt.tight_layout()


### Empty-mask policies are executable contracts

Binary region metrics in this notebook define empty/empty as a perfect agreement (`1.0`) and exactly-one-empty as no agreement (`0.0`). Boundary F1 uses the same policy. Per-class IoU keeps absent/absent classes as `NaN` so macro averaging excludes them explicitly. Other libraries may choose differently; record the policy with every aggregate.


In [ ]:
empty = np.zeros((12, 12), dtype=bool)
nonempty = empty.copy(); nonempty[3:8, 4:9] = True
empty_policy_cases = [
    ("GT empty, prediction empty", empty, empty, 1.0),
    ("GT empty, prediction non-empty", nonempty, empty, 0.0),
    ("GT non-empty, prediction empty", empty, nonempty, 0.0),
]
empty_policy_examples = []
for case, prediction, ground_truth, expected in empty_policy_cases:
    row = {
        "case": case,
        "IoU": binary_iou(prediction, ground_truth),
        "Dice": dice_score(prediction, ground_truth),
        "boundary_F1": boundary_f1(prediction, ground_truth, tolerance=1),
    }
    assert all(row[metric] == expected for metric in ("IoU", "Dice", "boundary_F1"))
    empty_policy_examples.append(row)
empty_policy_examples = pd.DataFrame(empty_policy_examples).set_index("case")
display(empty_policy_examples)


### Mask topology can fail while area overlap remains plausible

IoU and Dice count pixels, not components or holes. A continuous seal split into two pieces and a ring whose hole is filled have different operational meaning even when their overlap scores look usable. Component and hole counts are simple topology diagnostics—not a complete topological metric.


In [ ]:
def topology_signature(mask):
    mask = np.asarray(mask, bool)
    _, components = ndimage.label(mask)
    enclosed_background = ndimage.binary_fill_holes(mask) & ~mask
    _, holes = ndimage.label(enclosed_background)
    return {"components": int(components), "holes": int(holes)}


topology_y, topology_x = np.mgrid[:32, :32]
radius = np.sqrt((topology_x - 15.5) ** 2 + (topology_y - 15.5) ** 2)
seal_with_hole = (radius <= 11) & (radius >= 5)
broken_seal = seal_with_hole.copy(); broken_seal[:, 15:17] = False
filled_hole = ndimage.binary_fill_holes(seal_with_hole)

topology_examples = []
for name, prediction in (("correct topology", seal_with_hole), ("split into pieces", broken_seal), ("hole filled", filled_hole)):
    topology_examples.append({
        "prediction": name,
        "IoU": binary_iou(prediction, seal_with_hole),
        "Dice": dice_score(prediction, seal_with_hole),
        "boundary_F1": boundary_f1(prediction, seal_with_hole, tolerance=1),
        **topology_signature(prediction),
    })
topology_examples = pd.DataFrame(topology_examples).set_index("prediction")
assert topology_examples.loc["correct topology", "components"] == 1
assert topology_examples.loc["correct topology", "holes"] == 1
assert topology_examples.loc["split into pieces", "components"] == 2
assert topology_examples.loc["hole filled", "holes"] == 0
display(topology_examples.round(3))

fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, title, value in zip(axes, ["GT: continuous + hole", "wrong: two pieces", "wrong: hole filled"], [seal_with_hole, broken_seal, filled_hole]):
    ax.imshow(value, cmap="gray"); ax.set_title(title); ax.axis("off")
plt.tight_layout()


## 5. Trace a small U-Net

![U-Net encoder, bottleneck, decoder, and same-scale skips.](assets/unet-encoder-decoder.svg)

This network is intentionally small and readable. It is an educational architecture, not a state-of-the-art benchmark model.


In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, 3, padding=1), nn.BatchNorm2d(out_channels), nn.ReLU(),
        )
    def forward(self, x):
        return self.block(x)


class DownBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__(); self.pool = nn.MaxPool2d(2); self.conv = ConvBlock(in_channels, out_channels)
    def forward(self, x):
        return self.conv(self.pool(x))


class UpBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = ConvBlock(out_channels + skip_channels, out_channels)
    def forward(self, x, skip):
        x = self.up(x)
        assert x.shape[-2:] == skip.shape[-2:]
        return self.conv(torch.cat([x, skip], dim=1))


class TinyUNet(nn.Module):
    def __init__(self, classes=5, base=8):
        super().__init__()
        self.encoder_high = ConvBlock(3, base)
        self.encoder_low = DownBlock(base, base * 2)
        self.bottleneck = DownBlock(base * 2, base * 4)
        self.decoder_low = UpBlock(base * 4, base * 2, base * 2)
        self.decoder_high = UpBlock(base * 2, base, base)
        self.segmentation_head = nn.Conv2d(base, classes, 1)
        self.trace = {}
    def forward(self, x):
        e1 = self.encoder_high(x); e2 = self.encoder_low(e1); b = self.bottleneck(e2)
        d2 = self.decoder_low(b, e2); d1 = self.decoder_high(d2, e1)
        logits = self.segmentation_head(d1)
        self.trace = {name: tuple(value.shape) for name, value in {"input": x, "encoder_high": e1, "encoder_low": e2, "bottleneck": b, "decoder_low": d2, "decoder_high": d1, "logits": logits}.items()}
        assert logits.shape[0] == x.shape[0] and logits.shape[-2:] == x.shape[-2:]
        return logits


shape_model = TinyUNet(classes=len(CLASS_NAMES)).to(DEVICE)
with torch.inference_mode():
    shape_output = shape_model(torch.zeros(2, 3, IMAGE_SIZE, IMAGE_SIZE))
display(pd.Series(shape_model.trace, name="B × C × H × W").to_frame())
assert shape_output.shape == (2, len(CLASS_NAMES), IMAGE_SIZE, IMAGE_SIZE)
print(f"trainable parameters: {sum(p.numel() for p in shape_model.parameters()):,}")


## 6. Train with CE, Dice, and CE + Dice

All three runs use the same corpus, architecture, seed, optimizer, and epoch budget. This is a controlled teaching comparison, not a claim that two epochs identify a universally best loss.


In [ ]:
class SegmentationDataset(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, index):
        sample = self.samples[index]
        x = torch.from_numpy(sample["image"].copy()).permute(2, 0, 1).float() / 255.0
        y = torch.from_numpy(sample["mask"].copy()).long()
        return x, y, index


def multiclass_dice_loss(logits, target, include_background=True, eps=1e-6):
    probs = logits.softmax(dim=1)
    one_hot = F.one_hot(target, num_classes=logits.shape[1]).permute(0, 3, 1, 2).float()
    if not include_background:
        probs, one_hot = probs[:, 1:], one_hot[:, 1:]
    intersection = (probs * one_hot).sum(dim=(0, 2, 3))
    denominator = probs.sum(dim=(0, 2, 3)) + one_hot.sum(dim=(0, 2, 3))
    return 1 - ((2 * intersection + eps) / (denominator + eps)).mean()


train_pixel_counts = np.bincount(np.concatenate([sample["mask"].ravel() for sample in train_samples]), minlength=len(CLASS_NAMES))
train_pixel_frequencies = train_pixel_counts / train_pixel_counts.sum()
class_weights = 1 / np.sqrt(np.maximum(train_pixel_frequencies, 1e-6))
class_weights = torch.tensor(class_weights / class_weights.mean(), dtype=torch.float32, device=DEVICE)
print("class weights:", {CLASS_NAMES[i]: round(float(value), 3) for i, value in enumerate(class_weights)})


def compute_loss(logits, target, mode):
    ce = F.cross_entropy(logits, target, weight=class_weights)
    dice = multiclass_dice_loss(logits, target, include_background=True)
    return {"CE": ce, "Dice": dice, "CE+Dice": ce + dice}[mode]


def train_model(mode, epochs=6):
    torch.manual_seed(SEED)
    model = TinyUNet(classes=len(CLASS_NAMES), base=8).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=6e-3)
    generator = torch.Generator().manual_seed(SEED)
    loader = DataLoader(SegmentationDataset(train_samples), batch_size=8, shuffle=True, generator=generator)
    history = []
    for epoch in range(epochs):
        model.train(); running = 0.0
        for images, masks, _ in loader:
            optimizer.zero_grad(set_to_none=True)
            loss = compute_loss(model(images.to(DEVICE)), masks.to(DEVICE), mode)
            loss.backward(); optimizer.step(); running += loss.item() * len(images)
        history.append(running / len(loader.dataset))
    return model.eval(), history


loss_models, loss_histories = {}, {}
for loss_name in ("CE", "Dice", "CE+Dice"):
    loss_models[loss_name], loss_histories[loss_name] = train_model(loss_name)
    print(loss_name, "loss curve:", [round(value, 4) for value in loss_histories[loss_name]])

fig, ax = plt.subplots(figsize=(6, 3.5))
for name, values in loss_histories.items(): ax.plot(range(1, len(values) + 1), values, marker="o", label=name)
ax.set(xlabel="epoch", ylabel="training objective", title="Objectives have different scales"); ax.legend(); ax.grid(alpha=.25); plt.show()


## 7. Compare mask evidence, not training loss

Evaluation uses full masks and exposes background dominance. The aggregate foreground Dice and recall collapse all foreground classes together; mIoU preserves class identities. Boundary F1 uses a one-pixel tolerance.


In [ ]:
def predict_mask(model, sample):
    x = torch.from_numpy(sample["image"].copy()).permute(2, 0, 1).float().unsqueeze(0) / 255.0
    with torch.inference_mode(): return model(x.to(DEVICE)).argmax(1)[0].cpu().numpy().astype(np.uint8)


def evaluate_prediction(pred, target):
    class_iou = per_class_iou(pred, target, CLASS_NAMES)
    rare_values = [class_iou[class_id] for class_id in (2, 3, 4) if not np.isnan(class_iou[class_id])]
    return {
        "pixel_accuracy": pixel_accuracy(pred, target),
        "mIoU": float(np.nanmean(list(class_iou.values()))),
        "foreground_Dice": dice_score(pred > 0, target > 0),
        "foreground_recall": foreground_recall(pred, target),
        "boundary_F1": boundary_f1(pred > 0, target > 0, tolerance=1),
        "rare_class_IoU": float(np.mean(rare_values)) if rare_values else np.nan,
    }


evaluation_rows = []
prediction_cache = {}
for loss_name, model in loss_models.items():
    for sample in test_samples:
        pred = predict_mask(model, sample); prediction_cache[(loss_name, sample["id"])] = pred
        evaluation_rows.append({"loss": loss_name, "id": sample["id"], "source": sample["source"], **{k: sample[k] for k in ("tiny", "thin", "low_contrast", "occlusion", "boundary_heavy")}, **evaluate_prediction(pred, sample["mask"])})
evaluation = pd.DataFrame(evaluation_rows)
loss_comparison = evaluation.groupby("loss")[["pixel_accuracy", "mIoU", "foreground_Dice", "foreground_recall", "boundary_F1", "rare_class_IoU"]].mean().sort_values("mIoU", ascending=False)
display(loss_comparison.round(3))

best_loss = str(loss_comparison.index[0])
best_model = loss_models[best_loss]
print("Provisional model for downstream diagnostics:", best_loss, "— selected on this synthetic evaluation only")


## 8. Failure slices and source shift

Factory C was never used for optimization. Slice counts remain visible so a single difficult example cannot masquerade as a stable rate.


In [ ]:
best_eval = evaluation[evaluation["loss"] == best_loss].copy()
slice_rows = []
for slice_name, selector in {
    "all": np.ones(len(best_eval), bool),
    "tiny": best_eval["tiny"].to_numpy(bool),
    "thin": best_eval["thin"].to_numpy(bool),
    "low_contrast": best_eval["low_contrast"].to_numpy(bool),
    "occlusion": best_eval["occlusion"].to_numpy(bool),
    "boundary_heavy": best_eval["boundary_heavy"].to_numpy(bool),
    "Factory_C": (best_eval["source"] == "C").to_numpy(),
}.items():
    subset = best_eval.loc[selector]
    slice_rows.append({"slice": slice_name, "count": len(subset), **subset[["mIoU", "foreground_Dice", "foreground_recall", "boundary_F1"]].mean().to_dict()})
slice_table = pd.DataFrame(slice_rows).set_index("slice")
display(slice_table.round(3))

source_shift = best_eval.groupby("source")[["mIoU", "foreground_Dice", "foreground_recall", "boundary_F1"]].agg(["mean", "std", "count"])
display(source_shift.round(3))

worst_ids = best_eval.nsmallest(4, "mIoU")["id"].tolist()
fig, axes = plt.subplots(len(worst_ids), 3, figsize=(9, 3 * len(worst_ids)))
for row, sample_id in enumerate(worst_ids):
    sample = next(s for s in test_samples if s["id"] == sample_id)
    pred = prediction_cache[(best_loss, sample_id)]
    for col, (title, value) in enumerate((("image", sample["image"]), ("target", colorize(sample["mask"])), ("prediction", colorize(pred)))):
        axes[row, col].imshow(value); axes[row, col].set_title(f"{sample_id} · {title}"); axes[row, col].axis("off")
plt.tight_layout()


## 9. Semantic masks do not preserve instance identity

Connected components recover separate instances only when same-class regions do not touch. The generated instance map records construction-time identities and makes the information loss visible.


In [ ]:
instance_sample = next(sample for sample in test_samples if len(np.unique(sample["instances"])) > 3)
semantic_component = instance_sample["mask"] == 1
recovered_instances, recovered_count = ndimage.label(semantic_component)
true_ids = [int(value) for value in np.unique(instance_sample["instances"]) if value > 0]
print({"semantic_class": "component", "connected_components_recovered": int(recovered_count), "stored_visible_instance_ids": true_ids})

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, title, value in zip(axes, ["semantic mask", "stored instance IDs", "connected components"], [semantic_component, instance_sample["instances"], recovered_instances]):
    ax.imshow(value, cmap="tab20"); ax.set_title(title); ax.axis("off")
plt.tight_layout()


## 10. A transparent local promptable baseline

The proxy groups pixels by color similarity and connectivity. A positive point chooses a component; a negative point removes a color-similar region; a box estimates background from its border. This is useful for testing prompt protocols, not for claiming foundation-model quality.

Every local prompt table and artifact carries `engine="local_prompt_proxy"` and `foundation_model=False`. Actual SAM observations live in a separate optional evidence partition.


In [ ]:
LOCAL_PROMPT_ENGINE = {"engine": "local_prompt_proxy", "foundation_model": False, "model_weights": None, "network_required": False}
print("local prompt engine contract:", LOCAL_PROMPT_ENGINE)


def largest_interior_point(mask):
    distance = ndimage.distance_transform_edt(np.asarray(mask, bool))
    y, x = np.unravel_index(np.argmax(distance), distance.shape)
    return int(x), int(y)


def connected_region_containing(candidate, point):
    x, y = point
    labels, _ = ndimage.label(candidate)
    label_id = labels[np.clip(y, 0, labels.shape[0] - 1), np.clip(x, 0, labels.shape[1] - 1)]
    return labels == label_id if label_id else np.zeros_like(candidate, bool)


def point_prompt_proxy(image, positive_point, negative_points=(), threshold=0.16, previous_mask=None):
    values = np.asarray(image, float) / 255.0
    x, y = positive_point
    x, y = int(np.clip(x, 0, values.shape[1] - 1)), int(np.clip(y, 0, values.shape[0] - 1))
    seed = values[y, x]
    distance = np.linalg.norm(values - seed, axis=2)
    candidate = ndimage.binary_closing(distance < threshold, iterations=2)
    candidate = ndimage.binary_opening(candidate, iterations=1)
    candidate = connected_region_containing(candidate, (x, y))
    if previous_mask is not None:
        candidate = np.logical_or(candidate, np.asarray(previous_mask, bool))
    for nx, ny in negative_points:
        nx, ny = int(np.clip(nx, 0, values.shape[1] - 1)), int(np.clip(ny, 0, values.shape[0] - 1))
        negative_distance = np.linalg.norm(values - values[ny, nx], axis=2)
        candidate &= ~(negative_distance < threshold * 0.8)
        disk = (np.indices(candidate.shape)[1] - nx) ** 2 + (np.indices(candidate.shape)[0] - ny) ** 2 <= 2**2
        candidate &= ~disk
    return ndimage.binary_fill_holes(candidate)


def box_from_mask(mask, pad=0):
    ys, xs = np.where(mask)
    return (max(0, int(xs.min()) - pad), max(0, int(ys.min()) - pad), min(mask.shape[1], int(xs.max()) + 1 + pad), min(mask.shape[0], int(ys.max()) + 1 + pad))


def box_prompt_proxy(image, box):
    values = np.asarray(image, float) / 255.0
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1), max(0, y1); x2, y2 = min(values.shape[1], x2), min(values.shape[0], y2)
    crop = values[y1:y2, x1:x2]
    if min(crop.shape[:2]) < 3: return np.zeros(values.shape[:2], bool)
    border = np.concatenate([crop[0], crop[-1], crop[:, 0], crop[:, -1]], axis=0)
    background = np.median(border, axis=0)
    distance = np.linalg.norm(crop - background, axis=2)
    positive = distance[distance > 0]
    threshold = max(0.055, float(np.quantile(positive, 0.42)) if positive.size else 1.0)
    candidate = distance > threshold
    candidate = ndimage.binary_closing(candidate, iterations=2)
    candidate = ndimage.binary_fill_holes(candidate)
    output = np.zeros(values.shape[:2], bool); output[y1:y2, x1:x2] = candidate
    return output


def stability_quality(mask):
    mask = np.asarray(mask, bool)
    eroded = ndimage.binary_erosion(mask); dilated = ndimage.binary_dilation(mask)
    return binary_iou(eroded, dilated) if dilated.any() else 0.0


prompt_sample = test_samples[1]
prompt_target = prompt_sample["mask"] == 1
positive_point = largest_interior_point(prompt_target)
negative_candidates = np.argwhere(prompt_sample["mask"] == 2)
negative_point = tuple(negative_candidates[len(negative_candidates) // 2][::-1])
point_mask = point_prompt_proxy(prompt_sample["image"], positive_point, threshold=0.22)
refined_mask = point_prompt_proxy(prompt_sample["image"], positive_point, [negative_point], threshold=0.22, previous_mask=point_mask)
box = box_from_mask(prompt_target, pad=1)
box_mask = box_prompt_proxy(prompt_sample["image"], box)

prompt_contract = pd.DataFrame([
    {**LOCAL_PROMPT_ENGINE, "prompt": "positive point", **evaluate_prediction(point_mask.astype(np.uint8), prompt_target.astype(np.uint8))},
    {**LOCAL_PROMPT_ENGINE, "prompt": "+ negative refinement", **evaluate_prediction(refined_mask.astype(np.uint8), prompt_target.astype(np.uint8))},
    {**LOCAL_PROMPT_ENGINE, "prompt": "oracle box", **evaluate_prediction(box_mask.astype(np.uint8), prompt_target.astype(np.uint8))},
]).set_index("prompt")
display(prompt_contract[["mIoU", "foreground_Dice", "boundary_F1"]].round(3))

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for ax, title, value in zip(axes, ["image + prompts", "point", "+ negative", "box"], [prompt_sample["image"], point_mask, refined_mask, box_mask]):
    ax.imshow(value, cmap="gray" if value.ndim == 2 else None); ax.set_title(title); ax.axis("off")
axes[0].scatter(*positive_point, c="lime", s=50, edgecolors="black"); axes[0].scatter(*negative_point, c="red", s=50, edgecolors="white")
axes[0].add_patch(plt.Rectangle((box[0], box[1]), box[2]-box[0], box[3]-box[1], fill=False, color="cyan", linewidth=2))
plt.tight_layout()


## 11. Point and box prompts answer different target questions

The point uses local appearance and connectivity. The box supplies extent but may include several materials. The box here is derived from ground truth and is therefore an **oracle prompt**, not detector evidence.


In [ ]:
prompt_rows = []
for sample in test_samples[:8] + test_samples[-4:]:
    target = sample["mask"] == 1
    point = largest_interior_point(target)
    point_pred = point_prompt_proxy(sample["image"], point)
    oracle_box = box_from_mask(target, pad=1)
    box_pred = box_prompt_proxy(sample["image"], oracle_box)
    for prompt_type, pred in (("point", point_pred), ("oracle_box", box_pred)):
        prompt_rows.append({**LOCAL_PROMPT_ENGINE, "id": sample["id"], "source": sample["source"], "prompt": prompt_type, "information_budget": "one target point" if prompt_type == "point" else "ground-truth-derived oracle box", "IoU": binary_iou(pred, target), "Dice": dice_score(pred, target), "boundary_F1": boundary_f1(pred, target, 1), "quality_estimate": stability_quality(pred)})
prompt_results = pd.DataFrame(prompt_rows)
display(prompt_results.groupby(["prompt", "source"])[["IoU", "Dice", "boundary_F1"]].agg(["mean", "std", "count"]).round(3))


## 12. Prompt perturbation and mask-quality reliability

We move the same positive point and compare a morphology-based **reported mask-quality proxy** with actual target IoU and boundary F1. The notebook reports Spearman rank correlation and quality buckets. This score is a ranking diagnostic, not a probability and not a calibrated IoU estimate.


In [ ]:
sensitivity_rows = []
for sample in test_samples[:6] + test_samples[-6:]:
    target = sample["mask"] == 1
    x, y = largest_interior_point(target)
    for distance in (0, 2, 5, 10):
        moved = (min(IMAGE_SIZE - 1, x + distance), y)
        pred = point_prompt_proxy(sample["image"], moved)
        sensitivity_rows.append({
            **LOCAL_PROMPT_ENGINE, "id": sample["id"], "source": sample["source"],
            "distance_px": distance, "IoU": binary_iou(pred, target),
            "Dice": dice_score(pred, target), "boundary_F1": boundary_f1(pred, target, 1),
            "area_fraction": float(pred.mean()), "reported_mask_quality": stability_quality(pred),
        })
prompt_sensitivity = pd.DataFrame(sensitivity_rows)
display(prompt_sensitivity.groupby("distance_px")[["IoU", "Dice", "boundary_F1", "area_fraction"]].agg(["mean", "std"]).round(3))

quality_rank_correlations = {
    "reported_quality_vs_actual_IoU_spearman": float(stats.spearmanr(prompt_sensitivity["reported_mask_quality"], prompt_sensitivity["IoU"]).statistic),
    "reported_quality_vs_boundary_F1_spearman": float(stats.spearmanr(prompt_sensitivity["reported_mask_quality"], prompt_sensitivity["boundary_F1"]).statistic),
}
bucketed = prompt_sensitivity.assign(quality_bucket=pd.qcut(prompt_sensitivity["reported_mask_quality"], q=3, duplicates="drop"))
quality_calibration_buckets = bucketed.groupby("quality_bucket", observed=True).agg(
    reported_quality_mean=("reported_mask_quality", "mean"),
    actual_IoU_mean=("IoU", "mean"),
    boundary_F1_mean=("boundary_F1", "mean"),
    count=("IoU", "size"),
).reset_index()
quality_calibration_buckets["quality_bucket"] = quality_calibration_buckets["quality_bucket"].astype(str)
print("rank correlations:", {key: round(value, 3) for key, value in quality_rank_correlations.items()})
display(quality_calibration_buckets.round(3))
print("Reported mask quality is not a probability and is not ground-truth IoU.")

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
summary = prompt_sensitivity.groupby("distance_px")["IoU"].agg(["mean", "std"])
axes[0].errorbar(summary.index, summary["mean"], yerr=summary["std"].fillna(0), marker="o", capsize=3); axes[0].set(xlabel="point movement (px)", ylabel="mask IoU", title="Prompt sensitivity"); axes[0].grid(alpha=.25)
axes[1].scatter(prompt_sensitivity["reported_mask_quality"], prompt_sensitivity["IoU"], c=prompt_sensitivity["boundary_F1"], cmap="viridis"); axes[1].set(xlabel="reported mask quality", ylabel="actual IoU", title="Quality score is not truth"); plt.tight_layout()


### Correction effort is an operating metric

A simulated reviewer uses ground truth to choose the largest current false-positive or false-negative region, then adds one negative or positive point. This is an **oracle reviewer simulation**: it measures the interaction protocol under controlled corrections, not real reviewer productivity. We report the initial prompt, two correction opportunities, and prompts required to reach IoU `0.90`.


In [ ]:
def simulated_reviewer_correction(image, current_mask, target_mask):
    current, target = np.asarray(current_mask, bool), np.asarray(target_mask, bool)
    false_positive, false_negative = current & ~target, target & ~current
    if false_positive.sum() >= false_negative.sum() and false_positive.any():
        correction_point = largest_interior_point(false_positive)
        values = np.asarray(image, float) / 255.0
        x, y = correction_point
        remove = np.linalg.norm(values - values[y, x], axis=2) < 0.13
        updated = current & ~remove
        correction_type = "negative point"
    elif false_negative.any():
        correction_point = largest_interior_point(false_negative)
        updated = current | point_prompt_proxy(image, correction_point, threshold=0.16)
        correction_type = "positive point"
    else:
        correction_point, correction_type, updated = None, "none needed", current
    return ndimage.binary_fill_holes(updated), correction_type, correction_point


TARGET_REVIEW_IOU = 0.90
correction_effort_rows = []
for sample in test_samples[:6] + test_samples[-6:]:
    target = sample["mask"] == 1
    initial_point = largest_interior_point(target)
    current = point_prompt_proxy(sample["image"], initial_point, threshold=0.22)
    prompt_trace = [{"type": "positive point", "point": initial_point}]
    for prompt_count in (1, 2, 3):
        correction_effort_rows.append({
            **LOCAL_PROMPT_ENGINE, "id": sample["id"], "source": sample["source"],
            "prompt_count": prompt_count, "IoU": binary_iou(current, target),
            "boundary_F1": boundary_f1(current, target, 1), "prompt_trace": list(prompt_trace),
        })
        if prompt_count < 3:
            current, correction_type, correction_point = simulated_reviewer_correction(sample["image"], current, target)
            prompt_trace.append({"type": correction_type, "point": correction_point})

correction_effort = pd.DataFrame(correction_effort_rows)
correction_effort_by_step = correction_effort.groupby("prompt_count")[["IoU", "boundary_F1"]].agg(["mean", "std", "count"])
prompts_to_target = correction_effort[correction_effort["IoU"] >= TARGET_REVIEW_IOU].groupby("id")["prompt_count"].min()
correction_effort_summary = {
    "target_IoU": TARGET_REVIEW_IOU,
    "samples": int(correction_effort["id"].nunique()),
    "success_rate_within_3_prompts": float(prompts_to_target.size / correction_effort["id"].nunique()),
    "median_prompts_for_successful_samples": float(prompts_to_target.median()) if len(prompts_to_target) else None,
    "reviewer_simulation": "oracle: uses target mask to select correction location",
}
display(correction_effort_by_step.round(3))
print(correction_effort_summary)


## 13. Detector box error propagates into masks

The perfect box is an oracle ceiling. Expand, shrink, and shift operations simulate localization error. The prompt policy and coordinate convention are explicit.


In [ ]:
def perturb_box(box, mode, amount, size=IMAGE_SIZE):
    x1, y1, x2, y2 = box
    if mode == "expand": values = (x1-amount, y1-amount, x2+amount, y2+amount)
    elif mode == "shrink": values = (x1+amount, y1+amount, x2-amount, y2-amount)
    elif mode == "shift": values = (x1+amount, y1, x2+amount, y2)
    else: values = box
    ax1, ay1, ax2, ay2 = values
    ax1, ay1 = np.clip([ax1, ay1], 0, size-2); ax2, ay2 = np.clip([ax2, ay2], [ax1+1, ay1+1], size)
    return tuple(int(v) for v in (ax1, ay1, ax2, ay2))


box_variants = [("perfect", "none", 0), ("expand_5", "expand", 5), ("expand_10", "expand", 10), ("shift_5", "shift", 5), ("shift_10", "shift", 10), ("undersized_5", "shrink", 5)]
box_rows = []
for sample in test_samples[:6] + test_samples[-6:]:
    target = sample["mask"] == 1; perfect_box = box_from_mask(target)
    for name, mode, amount in box_variants:
        candidate_box = perturb_box(perfect_box, mode, amount)
        pred = box_prompt_proxy(sample["image"], candidate_box)
        box_rows.append({"id": sample["id"], "source": sample["source"], "variant": name, "IoU": binary_iou(pred, target), "Dice": dice_score(pred, target), "boundary_F1": boundary_f1(pred, target, 1), "box": candidate_box})
box_error_propagation = pd.DataFrame(box_rows)
display(box_error_propagation.groupby("variant")[["IoU", "Dice", "boundary_F1"]].agg(["mean", "std"]).round(3))

box_summary = box_error_propagation.groupby("variant")["IoU"].agg(["mean", "std"]).reindex([name for name, _, _ in box_variants])
ax = box_summary["mean"].plot(kind="bar", yerr=box_summary["std"], capsize=3, figsize=(8, 3.5), title="Detection localization error → prompt error → mask degradation")
ax.set(ylabel="mask IoU", xlabel="box prompt"); ax.grid(axis="y", alpha=.25); plt.tight_layout()


## 14. Separate operating contracts—not a model leaderboard

The U-Net predicts all five fixed-taxonomy classes automatically from the image. The local proxy receives target information through a ground-truth-derived oracle box and predicts one requested region. The table keeps `information_budget` and `prompt_contract` visible. Its rows diagnose two operating contracts; they must not be ranked as if they received equivalent inputs or produced equivalent outputs.


In [ ]:
comparison_rows = []
for sample in test_samples:
    target = sample["mask"] == 1
    unet_component = prediction_cache[(best_loss, sample["id"])] == 1
    proxy = box_prompt_proxy(sample["image"], box_from_mask(target, pad=1))
    contracts = (
        ("task_specific_UNet", unet_component, "none", "image only; automatic fixed taxonomy", False),
        ("local_prompt_proxy", proxy, "oracle_box", "image + ground-truth-derived target box", False),
    )
    for system, pred, prompt_contract_name, information_budget, foundation_model in contracts:
        comparison_rows.append({
            "id": sample["id"], "source": sample["source"], "system": system,
            "prompt_contract": prompt_contract_name, "information_budget": information_budget,
            "foundation_model": foundation_model, "IoU": binary_iou(pred, target),
            "boundary_F1": boundary_f1(pred, target, 1),
        })
system_comparison = pd.DataFrame(comparison_rows)
display(system_comparison.groupby(["system", "prompt_contract", "information_budget", "source"])[["IoU", "boundary_F1"]].agg(["mean", "std", "count"]).round(3))
print("Interpret this as a contract comparison, not an apples-to-apples model ranking.")


## 15. Optional official SAM 3.1 adapter—disabled by default

The factual contract was rechecked on 2026-09-01 against the official repository at `660a5e9e1b8b4c02c0ad97229b88a09a6e4ff5b7`, its release notes, its SAM License, and the gated `facebook/sam3.1` model at revision `daa63191845a41281374e725f4c9e51c7a824460`.

SAM 3 supplies the 848M-parameter text/exemplar/visual-prompt architecture. The March 27, 2026 **SAM 3.1** update specifically adds Object Multiplex, new checkpoints, and optimized multi-object video inference. The adapter never installs, authenticates, downloads, or executes remote code. Opt-in observations must log the reported mask quality, actual IoU, boundary F1, prompt, repository revision, model revision, and checkpoint hash.


In [ ]:
SAM31_REPO_REVISION = "660a5e9e1b8b4c02c0ad97229b88a09a6e4ff5b7"
SAM31_MODEL_ID = "facebook/sam3.1"
SAM31_MODEL_REVISION = "daa63191845a41281374e725f4c9e51c7a824460"
SAM31_FACT_CHECK = {
    "reviewed_on": "2026-09-01",
    "base_model": "SAM 3: 848M parameters; text, exemplar, and visual prompt capabilities",
    "sam31_release": "2026-03-27",
    "sam31_delta": "Object Multiplex, new checkpoints, and optimized multi-object video inference",
    "license": "SAM License, last updated 2025-11-19; custom terms",
    "checkpoint_access": "manual gated access on Hugging Face",
}
RUN_SAM31 = os.getenv("RUN_SAM31", "0") == "1"
sam31_quality_observations = []


def file_sha256(file_path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(file_path, "rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def record_sam31_quality_observation(prompt_id, reported_mask_quality, predicted_mask, target_mask):
    row = {
        "engine": "sam31_official_optional", "foundation_model": True,
        "prompt_id": prompt_id, "reported_mask_quality": float(reported_mask_quality),
        "actual_IoU": binary_iou(predicted_mask, target_mask),
        "boundary_F1": boundary_f1(predicted_mask, target_mask, tolerance=1),
    }
    sam31_quality_observations.append(row)
    return row


if RUN_SAM31:
    declared_revision = os.environ.get("SAM3_REPO_REVISION")
    declared_model_revision = os.environ.get("SAM3_MODEL_REVISION")
    checkpoint_path = Path(os.environ["SAM3_CHECKPOINT"])
    assert declared_revision == SAM31_REPO_REVISION, "Refusing an unreviewed SAM 3 repository revision"
    assert declared_model_revision == SAM31_MODEL_REVISION, "Refusing an unreviewed SAM 3.1 model revision"
    assert checkpoint_path.is_file(), "Provide an explicit local checkpoint; implicit downloads are disabled"
    checkpoint_sha256 = file_sha256(checkpoint_path)
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.box_ops import box_xywh_to_cxcywh
    from sam3.model.sam3_image_processor import Sam3Processor

    sam31_model = build_sam3_image_model(checkpoint_path=str(checkpoint_path), load_from_HF=False, device="cuda")
    sam31_processor = Sam3Processor(sam31_model, confidence_threshold=0.5)

    def sam31_box_adapter(pil_image, xyxy_box):
        width, height = pil_image.size
        x1, y1, x2, y2 = [float(v) for v in xyxy_box]
        xywh = torch.tensor([[x1, y1, x2 - x1, y2 - y1]], device="cuda")
        cxcywh = box_xywh_to_cxcywh(xywh)
        cxcywh[:, [0, 2]] /= width; cxcywh[:, [1, 3]] /= height
        state = sam31_processor.set_image(pil_image)
        sam31_processor.reset_all_prompts(state)
        return sam31_processor.add_geometric_prompt(state=state, box=cxcywh.flatten().tolist(), label=True)

    sam31_adapter = {
        "status": "loaded", "engine": "sam31_official_optional", "foundation_model": True,
        "repo_revision": declared_revision, "model_id": SAM31_MODEL_ID,
        "model_revision": declared_model_revision, "checkpoint_sha256": checkpoint_sha256,
        "license": SAM31_FACT_CHECK["license"], "fact_check": SAM31_FACT_CHECK,
    }
else:
    sam31_adapter = {
        "status": "not_run", "engine": "sam31_official_optional", "foundation_model": True,
        "reason": "gated checkpoint and CUDA environment are outside default validation",
        "model_id": SAM31_MODEL_ID, "repo_revision": SAM31_REPO_REVISION,
        "model_revision": SAM31_MODEL_REVISION, "license": SAM31_FACT_CHECK["license"],
        "fact_check": SAM31_FACT_CHECK,
    }
print(sam31_adapter)


## 16. Human-review policy and evidence artifact

The demonstration thresholds below are **workflow examples for this notebook only**, not validated hardware or safety targets. A production policy must be selected against calibrated quality, reviewer capacity, error costs, and the intended physical consequence.


In [ ]:
def json_ready(value):
    if isinstance(value, dict): return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)): return [json_ready(v) for v in value]
    if isinstance(value, pd.DataFrame): return json_ready(value.reset_index().to_dict(orient="records"))
    if isinstance(value, pd.Series): return json_ready(value.to_dict())
    if isinstance(value, np.ndarray): return json_ready(value.tolist())
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.floating,)): return None if np.isnan(value) else float(value)
    if isinstance(value, float) and math.isnan(value): return None
    return value


human_review_policy = {
    "status": "demonstration_only",
    "automatic_accept_candidate_if": {"quality_estimate_min": 0.70, "source_in_validated_set": True},
    "route_to_reviewer_if": ["quality estimate below threshold", "unseen source", "tiny or thin critical region", "mask touches forbidden boundary"],
    "retain": ["original prediction", "model and checkpoint version", "prompt coordinates and order", "quality estimate", "reviewer edits", "final approval"],
    "physical_action": "prohibited without an independently validated downstream policy",
}

local_evidence = {
    "dataset_contract": {"type": "deterministic_synthetic", "image_shape": [IMAGE_SIZE, IMAGE_SIZE, 3], "train_sources": ["A", "B"], "held_out_source": "C", "train_count": len(train_samples), "test_count": len(test_samples), "seed": SEED},
    "mask_schema": {"dtype": "uint8", "shape": [IMAGE_SIZE, IMAGE_SIZE], "class_ids": CLASS_NAMES, "ignore_label": None, "resize": "nearest-neighbor only", "instance_map": "separate int32 visible-instance IDs"},
    "model_versions": {**VERSIONS, "task_model": f"TinyUNet(base=8, selected_loss={best_loss}, epochs=6)", "prompt_system": LOCAL_PROMPT_ENGINE},
    "promptable_system_contract": LOCAL_PROMPT_ENGINE,
    "loss_comparison": loss_comparison,
    "semantic_metrics": evaluation.groupby(["loss", "source"])[["pixel_accuracy", "mIoU", "foreground_Dice", "foreground_recall"]].mean(),
    "boundary_metrics": evaluation.groupby(["loss", "source"])["boundary_F1"].mean(),
    "empty_mask_policy": {"binary_empty_empty": 1.0, "binary_exactly_one_empty": 0.0, "absent_per_class_IoU": "NaN/excluded", "assertion_examples": empty_policy_examples},
    "topology_examples": topology_examples,
    "size_slices": slice_table,
    "source_shift": source_shift,
    "prompt_sensitivity": prompt_sensitivity.groupby("distance_px")[["IoU", "Dice", "boundary_F1", "area_fraction", "reported_mask_quality"]].agg(["mean", "std"]),
    "quality_calibration": {"engine": LOCAL_PROMPT_ENGINE, "rank_correlations": quality_rank_correlations, "buckets": quality_calibration_buckets, "warning": "reported quality is not probability or ground-truth IoU"},
    "correction_effort": {"summary": correction_effort_summary, "by_prompt_count": correction_effort_by_step, "observations": correction_effort},
    "box_prompt_error_propagation": box_error_propagation.groupby("variant")[["IoU", "Dice", "boundary_F1"]].agg(["mean", "std"]),
    "operating_contract_comparison_not_a_ranking": system_comparison.groupby(["system", "prompt_contract", "information_budget", "source"])[["IoU", "boundary_F1"]].mean(),
    "human_review_policy": human_review_policy,
    "limitations": [
        "Procedural images do not establish real inspection validity.",
        "Six training epochs favor runtime and pedagogy over full convergence.",
        "The local prompt proxy is not SAM and receives oracle prompts in declared experiments.",
        "Boundary F1 tolerance is one synthetic pixel, not a calibrated physical tolerance.",
        "No latency, memory, privacy, or reviewer-time claim has been validated on target systems.",
    ],
}

evidence = {
    "locally_measured_evidence": local_evidence,
    "optional_downloaded_model_observations": {"sam31": {"adapter": sam31_adapter, "quality_evaluation_contract": {"required_fields": ["reported_mask_quality", "actual_IoU", "boundary_F1", "prompt_id"], "analysis": ["Spearman rank correlation", "quality buckets"]}, "observations": sam31_quality_observations}},
    "unresolved_production_assumptions": [
        "real factory annotation agreement and leakage-safe grouping",
        "target-hardware latency, memory, and energy",
        "SAM License approval for the intended use",
        "calibrated mask-quality and human-review thresholds",
        "physical-unit boundary tolerance and action authorization",
    ],
}

artifact_dir = Path(".artifacts"); artifact_dir.mkdir(exist_ok=True)
artifact_path = artifact_dir / "course-06-segmentation-evidence.json"
artifact_path.write_text(json.dumps(json_ready(evidence), indent=2, sort_keys=True) + "\n", encoding="utf-8")
print(f"saved {artifact_path} ({artifact_path.stat().st_size:,} bytes)")
print("top-level evidence partitions:", list(evidence))


## 17. What you should now be able to explain without code

- Why can an all-background mask look accurate under pixel accuracy?
- Why do class masks require nearest-neighbor resize?
- How do residual connections differ from U-Net skip connections?
- Why do IoU and boundary F1 answer different questions?
- Why can a semantic mask lose instance identity?
- Why can one positive point have several valid mask interpretations?
- Why is a model-reported mask-quality score not ground-truth IoU?
- Why is a perfect-box prompt an oracle rather than detector evidence?
- When does a task-specific U-Net fit better than a promptable foundation model?
- What prompt and review provenance must survive into an approved mask?
- Why does a precise contour not prove semantic correctness?
- What evidence is missing before segmentation may drive physical action?

Course 06 asked **which pixels belong to this region**. Course 07 asks how to learn a visual space where similar objects, defects, and regions can be found efficiently.
